In [6]:
import requests
import polars as pl
import json
from openpyxl import Workbook
from datetime import datetime

In [7]:
params = {
    "serviceName": ["Azure Cognitive Search"],
    "productName": ["Azure AI Search"],
    "location": ["US West", "US Central"],
    "currencyCode": ["USD"],
    "type": ["Consumption"],
}

clean_url = "https://prices.azure.com/api/retail/prices"

timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
file_name = f"costs_summary_{timestamp}"
excel_file_name = f"{file_name}.xlsx"

def filter_values(value):
    if value in (None, ""):
        return []
    if isinstance(value, (list, tuple, set)):
        return [item for item in value if item not in (None, "")]
    return [value]


def escape_odata_value(value):
    return str(value).replace("'", "''")


filters = []
for key, value in params.items():
    value_filters = [f"{key} eq '{escape_odata_value(item)}'" for item in filter_values(value)]
    if len(value_filters) == 1:
        filters.append(value_filters[0])
    elif len(value_filters) > 1:
        filters.append(f"({' or '.join(value_filters)})")

new_url = clean_url + '?$filter=' + ' and '.join(filters)
print(new_url)

response_text = requests.get(new_url).text

https://prices.azure.com/api/retail/prices?$filter=serviceName eq 'Azure Cognitive Search' and productName eq 'Azure AI Search' and (location eq 'US West' or location eq 'US Central') and currencyCode eq 'USD' and type eq 'Consumption'


In [8]:
data = json.loads(response_text)
df = pl.DataFrame(data["Items"])

columns = ['location', 'retailPrice', 'serviceName', 'productName', 'meterName', 'skuName', 'armSkuName', 'unitOfMeasure']
df_values = df.select(columns).sort(['location', 'productName', 'skuName', 'unitOfMeasure'])

wb = Workbook()
ws = wb.active
ws.append(df_values.columns)
for row in df_values.iter_rows():
    ws.append(row)

ws.auto_filter.ref = ws.dimensions
ws.freeze_panes = "A2"

for column_cells in ws.columns:
    max_length = 0
    column_letter = column_cells[0].column_letter

    for cell in column_cells:
        if cell.value is not None:
            max_length = max(max_length, len(str(cell.value)))

    ws.column_dimensions[column_letter].width = max_length + 0

wb.save(excel_file_name)

with open(f"{file_name}.json", "w", encoding="utf-8") as f:
    f.write(response_text)

In [4]:
df

currencyCode,tierMinimumUnits,retailPrice,unitPrice,armRegionName,location,effectiveStartDate,meterId,meterName,productId,skuId,productName,skuName,serviceName,serviceId,serviceFamily,unitOfMeasure,type,isPrimaryMeterRegion,armSkuName
str,f64,f64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,bool,str
"""USD""",0.0,0.0,0.0,"""westus""","""US West""","""2026-01-01T00:00:00Z""","""0a0fc5d3-e457-5180-bead-da40cd…","""Free Agentic Retrieval Medium …","""DZH318Z0BQKZ""","""DZH318Z0BQKZ/02VV""","""Azure AI Search""","""Free Agentic Retrieval Medium …","""Azure Cognitive Search""","""DZH318X5DGJB""","""Web""","""1K""","""Consumption""",true,"""Free Agentic Retrieval Medium …"
"""USD""",0.0,1.344,1.344,"""centralus""","""US Central""","""2015-05-01T00:00:00Z""","""99f367c4-70a5-4595-a7f0-fb3847…","""Standard S2 Unit""","""DZH318Z0BQKZ""","""DZH318Z0BQKZ/005C""","""Azure AI Search""","""Standard S2""","""Azure Cognitive Search""","""DZH318X5DGJB""","""Web""","""1 Hour""","""Consumption""",false,""""""
"""USD""",0.0,8.4447,8.4447,"""centralus""","""US Central""","""2025-10-01T00:00:00Z""","""0a9a4eca-0510-55b9-94ca-84627c…","""Storage Optimized L2 CC Unit""","""DZH318Z0BQKZ""","""DZH318Z0BQKZ/01ZQ""","""Azure AI Search""","""Storage Optimized L2 CC""","""Azure Cognitive Search""","""DZH318X5DGJB""","""Web""","""1 Hour""","""Consumption""",true,"""Storage Optimized L2 CC"""
"""USD""",0.0,1.344,1.344,"""westus""","""US West""","""2015-05-01T00:00:00Z""","""99f367c4-70a5-4595-a7f0-fb3847…","""Standard S2 Unit""","""DZH318Z0BQKZ""","""DZH318Z0BQKZ/003X""","""Azure AI Search""","""Standard S2""","""Azure Cognitive Search""","""DZH318X5DGJB""","""Web""","""1 Hour""","""Consumption""",false,""""""
"""USD""",0.0,0.0,0.0,"""westus""","""US West""","""2015-05-01T00:00:00Z""","""ab5b133b-d9da-4c2d-be66-0c3d70…","""Free Unit""","""DZH318Z0BQKZ""","""DZH318Z0BQKZ/0035""","""Azure AI Search""","""Free""","""Azure Cognitive Search""","""DZH318X5DGJB""","""Web""","""1 Hour""","""Consumption""",false,""""""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""USD""",0.0,0.0,0.0,"""westus""","""US West""","""2026-01-01T00:00:00Z""","""f2c46ebd-bfd8-541f-84ac-8bbfff…","""Free Agentic Retrieval Low Rea…","""DZH318Z0BQKZ""","""DZH318Z0BQKZ/02VZ""","""Azure AI Search""","""Free Agentic Retrieval Low Rea…","""Azure Cognitive Search""","""DZH318X5DGJB""","""Web""","""1K""","""Consumption""",true,"""Free Agentic Retrieval Low Rea…"
"""USD""",0.0,1.0,1.0,"""centralus""","""US Central""","""2019-07-01T00:00:00Z""","""036fcc20-399f-4822-b568-d96c62…","""Document Cracking Image Extrac…","""DZH318Z0BQKZ""","""DZH318Z0BQKZ/006T""","""Azure AI Search""","""Document Cracking""","""Azure Cognitive Search""","""DZH318X5DGJB""","""Web""","""1K""","""Consumption""",false,""""""
"""USD""",1000.0,0.8,0.8,"""centralus""","""US Central""","""2019-07-01T00:00:00Z""","""036fcc20-399f-4822-b568-d96c62…","""Document Cracking Image Extrac…","""DZH318Z0BQKZ""","""DZH318Z0BQKZ/006T""","""Azure AI Search""","""Document Cracking""","""Azure Cognitive Search""","""DZH318X5DGJB""","""Web""","""1K""","""Consumption""",false,""""""
